In [1]:
# Repo-root bootstrap: makes source/ importable and data/models paths absolute
# regardless of where the notebook server was started.
import sys
import pathlib

try:
    ROOT = pathlib.Path(__file__).resolve().parents[2]
except NameError:  # __file__ not defined in some environments (e.g. Jupyter console)
    ROOT = pathlib.Path.cwd()
    while not (ROOT / 'source').exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'code' / '04_ml'))

# Exact factorial-HMM decoder for multi-channel CFTR recording

The patch clamp sums N iid copies of the 7-state CFTR kinetic chain, so the vector of occupancy counts is an exact Markov chain and the Bayes-optimal posterior over the open count is computable. This notebook validates the emissions/chain and reports the exact decoder on the frozen `synth_v1` benchmark (`benchmark.py` + `kinetics.py`).

In [2]:
import json
import time
import numpy as np

import benchmark as B
import kinetics as K

print('frozen dataset:', B.DEST)

frozen dataset: /home/morekaccino/Codes/masters-paper/data/derived/synth_v1


## 1. Validate emissions and the count-state chain

In [3]:
P0 = B.p0_matrix()
for N in (1, 3):
    grid = K.default_grid(N)
    h = grid[1] - grid[0]
    logE = K.emission_log_densities(N, grid)
    masses = [float(np.exp(logE[k]).sum() * h) for k in range(N + 1)]
    means = [float((np.exp(logE[k]) * grid).sum() * h) for k in range(N + 1)]
    print(f'N={N}: emission masses {np.round(masses, 6)}  means {np.round(means, 3)}')

states = K.count_states(2)
T2 = K.count_transition_matrix(2, P0)
print('N=2 chain row sums:', float(T2.sum(axis=1).min()), float(T2.sum(axis=1).max()))
pi = K.count_stationary(3, P0)
k = states = K.count_states(3)[:, B.STATEMAP == 1].sum(axis=1)
print('N=3 stationary mean open count:', round(float((pi * k).sum()), 4))

N=1: emission masses [1. 1.]  means [0.677 1.276]
N=3: emission masses [1. 1. 1. 1.]  means [2.03  2.63  3.229 3.828]
N=2 chain row sums: 0.9999999999999999 1.0000000000000004
N=3 stationary mean open count: 1.5923


## 2. Baseline: exact decoding with the true N on the frozen test set

In [4]:
cache = {}

def loaders(N, scale):
    key = (N, round(scale, 3))
    if key not in cache:
        g = K.default_grid(N, noise_scale=scale)
        cache[key] = (g, K.emission_log_densities(N, g, noise_scale=scale))
    return cache[key]

def decode_set(data, xkey, scale, n_traces=None):
    X = data[xkey]
    n = len(X) if n_traces is None else n_traces
    preds = np.stack([
        K.decode_open_count(X[i], int(data['N'][i]), grid=loaders(int(data['N'][i]), scale)[0],
                            logE=loaders(int(data['N'][i]), scale)[1])[0]
        for i in range(n)
    ])
    return preds

test = B.load_split('test')
results = {}
t0 = time.time()
for scale, xkey in [(1.0, 'X_s1'), (2.0, 'X_s2'), (4.0, 'X_s4')]:
    preds = decode_set(test, xkey, scale)
    m = B.summarize(test['y'], preds, N_true=test['N'])
    results[f'test_scale_{scale:g}'] = m
    print(f"scale x{scale:g}: acc={m['accuracy']:.4f}  macroF1={m['macro_f1']:.4f}  "
          f"transition={m['transition_accuracy']:.4f}  nontransition={m['non_transition_accuracy']:.4f}  "
          f"detection={m['transition_detection_rate']:.3f}  timing_err={m['mean_timing_error']:.2f}  "
          f"per-N={ {k: round(v['accuracy'],3) for k,v in m['per_n'].items()} }")
print(f'elapsed {time.time()-t0:.0f}s')

scale x1: acc=0.9625  macroF1=0.9533  transition=0.9276  nontransition=0.9956  detection=0.884  timing_err=1.39  per-N={1: 0.993, 2: 0.969, 3: 0.931}


scale x2: acc=0.8343  macroF1=0.8017  transition=0.7167  nontransition=0.9457  detection=0.603  timing_err=2.09  per-N={1: 0.953, 2: 0.838, 3: 0.732}


scale x4: acc=0.6715  macroF1=0.5759  transition=0.5448  nontransition=0.7915  detection=0.321  timing_err=2.57  per-N={1: 0.878, 2: 0.657, 3: 0.512}
elapsed 47s


## 3. N estimation from the marginal signal likelihood (iid binomial mixture)

In [5]:
t0 = time.time()
ok = 0
N_hat = np.empty(len(test['N']), dtype=int)
for i in range(len(test['X_s1'])):
    ll = K.signal_likelihood(test['X_s1'][i])
    N_hat[i] = max(ll, key=ll.get)
    ok += int(N_hat[i] == test['N'][i])
results['n_inference'] = B.n_estimation_metrics(test['N'], N_hat)
print('N exact accuracy:', round(results['n_inference']['n_accuracy'], 4),
      ' MAE:', round(results['n_inference']['n_mae'], 4), f'({time.time()-t0:.0f}s)')

N exact accuracy: 1.0  MAE: 0.0 (46s)


## 4. Extrapolation to N = 4 and 5 (400 traces, first 120 decoded for runtime)

In [6]:
extrap = B.load_split('extrap')
n_use = 120
preds = decode_set(extrap, 'X_s1', 1.0, n_traces=n_use)
m = B.summarize(extrap['y'][:n_use], preds, N_true=extrap['N'][:n_use])
results['extrap_scale_1'] = m
print('extrap x1:', round(m['accuracy'], 4), {k: round(v['accuracy'],3) for k,v in m['per_n'].items()})

ok = 0
for i in range(n_use):
    ll = K.signal_likelihood(extrap['X_s1'][i], N_values=range(1, 7))
    ok += int(max(ll, key=ll.get) == extrap['N'][i])
results['extrap_n_inference'] = {'n_accuracy': ok / n_use}
print('extrap N accuracy:', round(ok / n_use, 4))

extrap x1: 0.8646 {4: 0.888, 5: 0.843}


extrap N accuracy: 1.0


## 5. Save results

In [7]:
out = ROOT / 'code' / '04_ml' / 'results' / 'exact_hmm_synth_v1.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(results, indent=2))
print('saved', out)

saved /home/morekaccino/Codes/masters-paper/code/04_ml/results/exact_hmm_synth_v1.json
